# Windvorhersage für Basel: Schritt für Schritt

In diesem Notebook entwickeln wir eine Vorhersage der Windgeschwindigkeit in Basel eine Stunde im Voraus.

Wir laden historische Wetterdaten von Open-Meteo und berechnen zwei Eingaben für das Modell: den Windwert zum jeweiligen Zeitpunkt und den Durchschnitt der letzten sechs Stunden. Der Windwert eine Stunde später dient als Zielwert.

Anschliessend speichern wir die Daten in Hopsworks, beziehen Trainingsdaten über eine Feature View und trainieren einen Random Forest. Wir vergleichen seine Vorhersagen mit der einfachen Annahme, dass der Wind unverändert bleibt, und speichern das Modell.

Die Vorhersage mit neuen Daten ergänzen wir als nächsten Abschnitt. Danach übertragen wir den Ablauf in drei separate Pipelines: Feature-Pipeline, Trainings-Pipeline und Inferenz-Pipeline.

Die historischen Wetterwerte sind Reanalysedaten und keine direkten Messungen einer lokalen Wetterstation.


## 1. Projekt und Konfiguration laden

Diese Zelle lädt nur lokale Einstellungen.

In [1]:
from pathlib import Path
import json

project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Bitte das Notebook innerhalb des Projektordners starten.")

config = json.loads((project_root / "configs" / "basel.json").read_text())
config

{'latitude': 47.5596,
 'longitude': 7.5886,
 'timezone': 'Europe/Zurich',
 'forecast_horizon_hours': 1}

## 2. Python-Paket prüfen

Nach `uv sync` und Auswahl des Projekt-Kernels sollte dieser Import ohne zusätzliche Suchpfade funktionieren.

In [2]:
!pwd
import wind_basel

print("Das Projektpaket ist verfügbar.")

/Users/stephanberger/Documents/CAS-AI-Ops/cas-ai-operations-2026-mlops-project/notebooks
Das Projektpaket ist verfügbar.


## 3. Nächster gemeinsamer Schritt

Wir rufen einen kleinen Zeitraum mit Winddaten für Basel ab und betrachten die Tabelle. Erst danach legen wir Features und Zielwerte an.

Datenquellen: [Historische Wetterdaten](https://open-meteo.com/en/docs/historical-weather-api) und [aktuelle Wetterdaten](https://open-meteo.com/en/docs).

In [3]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": config["latitude"],
    "longitude": config["longitude"],
    "start_date": "2025-05-31",
    "end_date": "2025-06-07",
    "hourly": "wind_speed_10m",
    "wind_speed_unit": "kmh",
    "timezone": config["timezone"],
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

weather_data = response.json()

wind_df = pd.DataFrame(weather_data["hourly"])
wind_df["time"] = pd.to_datetime(wind_df["time"])

wind_df.head()

,time,wind_speed_10m
0,2025-05-31 00:00:00,3.1
1,2025-05-31 01:00:00,4.1
2,2025-05-31 02:00:00,3.6
3,2025-05-31 03:00:00,3.6
4,2025-05-31 04:00:00,3.2


In [4]:
wind_df.dtypes

time              datetime64[ns]
wind_speed_10m           float64
dtype: object

# Aggregiertes Feature: mittlere Windgeschwindigkeit über sechs Stunden

In [5]:
wind_df = wind_df.sort_values("time").copy()

wind_df["wind_mean_6h"] = (
    wind_df["wind_speed_10m"]
    .rolling(window=6, min_periods=6)
    .mean()
)

# Erst nach der Berechnung auf unseren gewünschten Zeitraum begrenzen
wind_df = wind_df.loc[
    wind_df["time"] >= "2025-06-01"
].copy()

wind_df.head(10)

,time,wind_speed_10m,wind_mean_6h
24,2025-06-01 00:00:00,2.0,4.383333
25,2025-06-01 01:00:00,7.2,5.233333
26,2025-06-01 02:00:00,5.2,4.533333
27,2025-06-01 03:00:00,4.2,4.850000
28,2025-06-01 04:00:00,7.7,5.066667
29,2025-06-01 05:00:00,2.9,4.866667
30,2025-06-01 06:00:00,4.6,5.300000
31,2025-06-01 07:00:00,6.5,5.183333
32,2025-06-01 08:00:00,2.7,4.766667
33,2025-06-01 09:00:00,4.6,4.833333


Umbenenen der Spalte wind_speed_10m für einfacheres verstäntniss

In [6]:
wind_df["wind_current"] = wind_df["wind_speed_10m"]

wind_df[["time", "wind_current", "wind_mean_6h"]].head()

,time,wind_current,wind_mean_6h
24,2025-06-01 00:00:00,2.0,4.383333
25,2025-06-01 01:00:00,7.2,5.233333
26,2025-06-01 02:00:00,5.2,4.533333
27,2025-06-01 03:00:00,4.2,4.850000
28,2025-06-01 04:00:00,7.7,5.066667


Hinzufügen der Label Spalte um das model zu trainieren.

In [7]:
wind_df = wind_df.sort_values("time").copy()

# Prüfen, dass die Zeilen genau eine Stunde auseinanderliegen
assert wind_df["time"].diff().dropna().eq(
    pd.Timedelta(hours=1)
).all(), "Die Zeitreihe enthält Lücken oder doppelte Stunden."

# Der nächste Stundenwert wird zum Zielwert dieser Zeile
wind_df["wind_next_hour"] = wind_df["wind_current"].shift(-1)

wind_df[
    ["time", "wind_current", "wind_mean_6h", "wind_next_hour"]
].head()

,time,wind_current,wind_mean_6h,wind_next_hour
24,2025-06-01 00:00:00,2.0,4.383333,7.2
25,2025-06-01 01:00:00,7.2,5.233333,5.2
26,2025-06-01 02:00:00,5.2,4.533333,4.2
27,2025-06-01 03:00:00,4.2,4.850000,7.7
28,2025-06-01 04:00:00,7.7,5.066667,2.9


Train,Test Split

In [8]:
feature_columns = ["wind_current", "wind_mean_6h"]
target_column = "wind_next_hour"

# Zeilen ohne vollständige Eingaben oder Zielwert weglassen
model_df = (
    wind_df
    .dropna(subset=feature_columns + [target_column])
    .sort_values("time")
    .copy()
)

split_at = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_at].copy()
test_df = model_df.iloc[split_at:].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("Trainingsbeispiele:", len(train_df))
print("Testbeispiele:", len(test_df))

Trainingsbeispiele: 133
Testbeispiele: 34


In [ ]:
import hopsworks
from getpass import getpass

project = hopsworks.login(
    host="eu-west.cloud.hopsworks.ai",
    project="wind_basel_stephan",
    api_key_value=getpass("Hopsworks API-Schlüssel: "),
)

fs = project.get_feature_store()
print("Mit dem Feature Store verbunden.")


⠸ h11==0.16.0                                                                   Resolved 106 packages in 458ms
Prepared 1 package in 407ms                                              ps/cas-ai⠹ Preparing packages... (0/1)                                                   
Uninstalled 1 package in 2ms
Installed 1 package in 1msfrom file:///Users/stephanberger/D
 ~ wind-basel==0.1.0 (from file:///Users/stephanberger/Documents/CAS-AI-Ops/cas-ai-operations-2026-mlops-project)


2026-10-04 18:49:15,818 INFO: Closing external client and cleaning up certificates.
2026-10-04 18:49:15,836 INFO: Connection closed.
2026-10-04 18:49:15,839 INFO: Initializing external client
2026-10-04 18:49:15,839 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


To ensure compatibility please install the latest bug fix release matching the minor version of your backend (5.1) by running 'pip install hopsworks==5.1.*'
2026-10-04 18:49:16,959 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/48254
Mit dem Feature Store verbunden.


In [12]:
feature_df = wind_df[
    ["time", "wind_current", "wind_mean_6h", "wind_next_hour"]
].dropna().copy()

feature_df["location"] = "basel"

# Basler Ortszeit in die weltweit eindeutige Zeit UTC umrechnen.
feature_df["time"] = (
    feature_df["time"]
    .dt.tz_localize("Europe/Zurich")
    .dt.tz_convert("UTC")
)

feature_df.head()

,time,wind_current,wind_mean_6h,wind_next_hour,location
24,2025-05-31 22:00:00+00:00,2.0,4.383333,7.2,basel
25,2025-05-31 23:00:00+00:00,7.2,5.233333,5.2,basel
26,2025-06-01 00:00:00+00:00,5.2,4.533333,4.2,basel
27,2025-06-01 01:00:00+00:00,4.2,4.850000,7.7,basel
28,2025-06-01 02:00:00+00:00,7.7,5.066667,2.9,basel


In [13]:
wind_fg = fs.get_or_create_feature_group(
    name="wind_basel",
    version=1,
    description="Stündliche Winddaten für Basel mit Features und Zielwert",
    primary_key=["location", "time"],
    event_time="time",
    online_enabled=False,
)

wind_fg.insert(feature_df)

(None, None)

In [16]:
# Zeit, zwei Modelleingaben und den Zielwert auswählen.
query = wind_fg.select([
    "time",
    "wind_current",
    "wind_mean_6h",
    "wind_next_hour",
])

feature_view = fs.get_or_create_feature_view(
    name="wind_basel_prediction",
    version=2,
    description="Windvorhersage mit Zeit als Hilfsspalte",
    query=query,
    labels=["wind_next_hour"],
    training_helper_columns=["time"],
)

print("Feature View ist bereit.")

Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/48254/fs/35913/fv/wind_basel_prediction/version/2
Feature View ist bereit.


In [17]:
features_df, labels_df = feature_view.training_data(
    description="Basler Winddaten mit Zeit als Hilfsspalte",
    training_helper_columns=True,
)

print(f"Geladene Zeilen: {len(features_df)}")
print("Spalten:", features_df.columns.tolist())

display(features_df.head())
display(labels_df.head())

2026-10-04 19:05:35,015 INFO: Computing insert statistics


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (0.64s) 
Geladene Zeilen: 167
Spalten: ['time', 'wind_current', 'wind_mean_6h']


,time,wind_current,wind_mean_6h
0,2025-05-31 22:00:00,2.0,4.383333
1,2025-05-31 23:00:00,7.2,5.233333
2,2025-06-01 00:00:00,5.2,4.533333
3,2025-06-01 01:00:00,4.2,4.850000
4,2025-06-01 02:00:00,7.7,5.066667


,wind_next_hour
0,7.2
1,5.2
2,4.2
3,7.7
4,2.9


In [19]:


# Eingaben und zugehörige Zielwerte gemeinsam zusammenführen.
training_df = features_df.reset_index(drop=True).copy()
training_df["wind_next_hour"] = (
    labels_df["wind_next_hour"].to_numpy()
)

# Gemeinsam sortieren, damit Eingaben und Zielwerte zusammenbleiben.
training_df = training_df.sort_values("time").reset_index(drop=True)

split_at = int(len(training_df) * 0.8)

train_df = training_df.iloc[:split_at].copy()
test_df = training_df.iloc[split_at:].copy()

feature_columns = ["wind_current", "wind_mean_6h"]
target_column = "wind_next_hour"

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print(f"Trainingsbeispiele: {len(X_train)}")
print(f"Testbeispiele: {len(X_test)}")

Trainingsbeispiele: 133
Testbeispiele: 34


In [20]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42,
)

model.fit(X_train, y_train)

print("Das Modell ist trainiert.")

Das Modell ist trainiert.


In [21]:
from sklearn.metrics import mean_absolute_error

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

print(f"Mittlerer absoluter Fehler: {mae:.2f} km/h")

Mittlerer absoluter Fehler: 3.18 km/h


In [22]:
# Basisannahme: Der Wind bleibt eine weitere Stunde gleich.
baseline_predictions = X_test["wind_current"]

baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"Basisannahme:  {baseline_mae:.2f} km/h")
print(f"Random Forest: {mae:.2f} km/h")

Basisannahme:  3.49 km/h
Random Forest: 3.18 km/h


In [23]:
import joblib

# Das trainierte Modell als Datei speichern.
model_dir = project_root / "models" / "wind_basel"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, model_dir / "model.joblib")

# Die Model Registry öffnen.
mr = project.get_model_registry()

# Modellbeschreibung und Fehlerwert hinterlegen.
registered_model = mr.python.create_model(
    name="wind_basel_random_forest",
    description="Wind in Basel eine Stunde voraus, mit zwei Wind-Features",
    metrics={"mae_kmh": float(mae)},
    feature_view=feature_view,
)

# Modelldatei nach Hopsworks hochladen.
registered_model.save(str(model_dir))

print("Modell in Hopsworks gespeichert.")

Uploading model files (0 dirs, 0 files):  17%|█▋        | 1/6 [00:01<00:05,  1.12s/it]

Moving model files from '/Users/stephanberger/Documents/CAS-AI-Ops/cas-ai-operations-2026-mlops-project/models/wind_basel' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading /Users/stephanberger/Documents/CAS-AI-Ops/cas-ai-operations-2026-mlops-project/models/wind_basel/model.joblib: 100.000%|██████████| 358689/358689 elapsed<00:00 remaining<00:00
Uploading /var/folders/m9/r_62zv1157lg1zmhvqzjrzp80000gn/T/tmp8hfghy7f/model_schema.json: 100.000%|██████████| 406/406 elapsed<00:00 remaining<00:00
Model export complete: 100%|██████████| 6/6 [00:09<00:00,  1.63s/it]                   

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/48254/models/wind_basel_random_forest/2
Modell in Hopsworks gespeichert.


## Inferenz: Wind eine Stunde im Voraus vorhersagen

Wir laden das gespeicherte Modell aus der Hopsworks Model Registry,
ohne es erneut zu trainieren.

Für die Vorhersage kombinieren wir zwei Eingaben:
- den aktuellen Windwert direkt aus der Open-Meteo-API;
- den Sechsstunden-Durchschnitt, den wir berechnen, in Hopsworks
  speichern und über die Feature View zurücklesen.

Der Durchschnitt endet an der letzten vollen Stunde. Der aktuelle
Windwert kann einen neueren Zeitpunkt haben. Diese zeitliche Abweichung
gegenüber den stündlichen Trainingsdaten ist eine Vereinfachung unseres
Prototyps. Die aktuellen API-Werte stammen aus Wettermodellen, nicht
aus direkten Messungen einer lokalen Wetterstation.

### Gespeichertes Modell laden

Wir laden die zuvor registrierte Modellversion herunter und öffnen
die Modelldatei mit Joblib.

In [25]:
# Die gerade registrierte Modellversion herunterladen.
downloaded_model_dir = registered_model.download()

# Das gespeicherte Modell laden.
inference_model = joblib.load(
    Path(downloaded_model_dir) / "model.joblib"
)

print("Gespeichertes Modell für die Inferenz geladen.")

Downloading: 100.000%|██████████| 358689/358689 elapsed<00:00 remaining<00:00

Gespeichertes Modell für die Inferenz geladen.. DONE


Aktuellen Wind Abrufen

In [27]:
current_response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": config["latitude"],
        "longitude": config["longitude"],
        "current": "wind_speed_10m",
        "hourly": "wind_speed_10m",
        "past_days": 1,
        "forecast_days": 1,
        "wind_speed_unit": "kmh",
        "timezone": "UTC",
    },
    timeout=30,
)
current_response.raise_for_status()
current_data = current_response.json()

# Zeitpunkt des aktuellen Windwerts eindeutig als UTC lesen.
current_time = pd.to_datetime(
    current_data["current"]["time"], utc=True
)

# Stundenwerte als Tabelle aufbereiten.
recent_wind_df = pd.DataFrame(current_data["hourly"])
recent_wind_df["time"] = pd.to_datetime(
    recent_wind_df["time"], utc=True
)

# Nur Zeitpunkte bis zum aktuellen Wert behalten.
recent_wind_df = (
    recent_wind_df.loc[recent_wind_df["time"] <= current_time]
    .sort_values("time")
    .copy()
)

display(recent_wind_df.tail(6))

,time,wind_speed_10m
36,2026-10-04 12:00:00+00:00,2.6
37,2026-10-04 13:00:00+00:00,4.8
38,2026-10-04 14:00:00+00:00,9.8
39,2026-10-04 15:00:00+00:00,11.6
40,2026-10-04 16:00:00+00:00,10.7
41,2026-10-04 17:00:00+00:00,8.1


In [28]:
# Letzte volle Stunde bis zum aktuellen Datenzeitpunkt.
aggregate_time = current_time.floor("h")

# Genau diese Stunde und die fünf vorherigen Stunden auswählen.
expected_times = pd.date_range(
    end=aggregate_time,
    periods=6,
    freq="h",
)

last_six_hours = (
    recent_wind_df
    .set_index("time")
    .reindex(expected_times)
)

# Bei fehlenden Werten keinen unvollständigen Mittelwert verwenden.
if last_six_hours["wind_speed_10m"].isna().any():
    raise ValueError("Für den Durchschnitt fehlen Stundenwerte.")

wind_mean_6h = float(last_six_hours["wind_speed_10m"].mean())

print(f"Sechsstunden-Durchschnitt: {wind_mean_6h:.2f} km/h")
print("Stand des Durchschnitts:", aggregate_time)

Sechsstunden-Durchschnitt: 7.93 km/h
Stand des Durchschnitts: 2026-10-04 17:00:00+00:00


In [29]:
live_feature_df = pd.DataFrame({
    "time": [aggregate_time],
    "wind_current": [
        float(last_six_hours.loc[aggregate_time, "wind_speed_10m"])
    ],
    "wind_mean_6h": [wind_mean_6h],
    "wind_next_hour": [float("nan")],
    "location": ["basel"],
})

display(live_feature_df)

,time,wind_current,wind_mean_6h,wind_next_hour,location
0,2026-10-04 17:00:00+00:00,8.1,7.933333,NaN,basel


In [30]:
# In die bestehende Feature Group schreiben.
# Warten, bis der Schreibvorgang abgeschlossen ist.
wind_fg.insert(live_feature_df, wait=True)

print("Aktuelle Feature-Zeile in Hopsworks gespeichert.")

Aktuelle Feature-Zeile in Hopsworks gespeichert.


In [31]:
stored_features = feature_view.get_batch_data(
    start_time=aggregate_time.to_pydatetime(),
    end_time=(aggregate_time + pd.Timedelta(hours=1)).to_pydatetime(),
)

display(stored_features)

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (0.85s) 


,wind_current,wind_mean_6h
0,8.1,7.933333


In [ ]:
# Für Basel und die gewählte Stunde erwarten wir genau eine Zeile.
if len(stored_features) != 1:
    raise ValueError("Genau eine Feature-Zeile aus Hopsworks erwartet.")

# Gleiche Eingabespalten und Reihenfolge wie beim Training.
prediction_input = pd.DataFrame({
    # Echtzeit-Feature: aktueller Wind direkt aus der API.
    "wind_current": [
        current_data["current"]["wind_speed_10m"]
    ],
    # Aggregiertes Feature: Sechsstunden-Durchschnitt aus Hopsworks.
    "wind_mean_6h": [
        stored_features["wind_mean_6h"].iloc[0]
    ],
})

if prediction_input.isna().any().any():
    raise ValueError("Mindestens ein Eingabewert fehlt.")

# Gespeichertes Modell anwenden.
predicted_wind = float(
    inference_model.predict(prediction_input)[0]
)

# Unser Modell wurde für eine Stunde im Voraus trainiert.
prediction_time = current_time + pd.Timedelta(hours=1)
local_prediction_time = prediction_time.tz_convert(config["timezone"])

display(prediction_input)
print(
    f"Vorhersage für {local_prediction_time:%d.%m.%Y %H:%M %Z}: "
    f"{predicted_wind:.2f} km/h"
)

,wind_current,wind_mean_6h
0,7.7,7.933333


Vorhersage für 04.10.2026 20:30 CEST: 5.67 km/h


In [33]:
feature_time = pd.to_datetime(live_feature_df["time"].iloc[0], utc=True)
label_time = feature_time + pd.Timedelta(hours=1)

print("Zeitpunkt der gespeicherten Zeile:", feature_time)
print("Gesuchter Windwert für:", label_time)

Zeitpunkt der gespeicherten Zeile: 2026-10-04 17:00:00+00:00
Gesuchter Windwert für: 2026-10-04 18:00:00+00:00


In [34]:
label_response = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": config["latitude"],
        "longitude": config["longitude"],
        "start_date": label_time.strftime("%Y-%m-%d"),
        "end_date": label_time.strftime("%Y-%m-%d"),
        "hourly": "wind_speed_10m",
        "wind_speed_unit": "kmh",
        "timezone": "UTC",
    },
    timeout=30,
)

print("HTTP-Status:", label_response.status_code)
display(label_response.json())

HTTP-Status: 200


{'latitude': 47.557117,
 'longitude': 7.549342,
 'generationtime_ms': 0.3629922866821289,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 247.0,
 'hourly_units': {'time': 'iso8601', 'wind_speed_10m': 'km/h'},
 'hourly': {'time': ['2026-10-04T00:00',
   '2026-10-04T01:00',
   '2026-10-04T02:00',
   '2026-10-04T03:00',
   '2026-10-04T04:00',
   '2026-10-04T05:00',
   '2026-10-04T06:00',
   '2026-10-04T07:00',
   '2026-10-04T08:00',
   '2026-10-04T09:00',
   '2026-10-04T10:00',
   '2026-10-04T11:00',
   '2026-10-04T12:00',
   '2026-10-04T13:00',
   '2026-10-04T14:00',
   '2026-10-04T15:00',
   '2026-10-04T16:00',
   '2026-10-04T17:00',
   '2026-10-04T18:00',
   '2026-10-04T19:00',
   '2026-10-04T20:00',
   '2026-10-04T21:00',
   '2026-10-04T22:00',
   '2026-10-04T23:00'],
  'wind_speed_10m': [2.1,
   4.3,
   2.5,
   1.5,
   0.5,
   0.4,
   0.4,
   1.3,
   3.2,
   2.0,
   4.2,
   4.6,
   6.0,
   6.2,
   4.6,
   6.6,
   9.8,
   6.5,
   6.4,
   6.

In [35]:
label_response.raise_for_status()

label_df = pd.DataFrame(label_response.json()["hourly"])
label_df["time"] = pd.to_datetime(label_df["time"], utc=True)

target_row = label_df.loc[label_df["time"] == label_time]

if len(target_row) != 1:
    raise ValueError("Die gesuchte Stunde fehlt oder ist mehrfach vorhanden.")

if target_row["wind_speed_10m"].isna().any():
    raise ValueError("Für diese Stunde ist noch kein Windwert verfügbar.")

label_value = float(target_row["wind_speed_10m"].iloc[0])
display(target_row)


,time,wind_speed_10m
18,2026-10-04 18:00:00+00:00,6.4


In [36]:
# Ursprüngliche Zeile kopieren und den Zielwert ergänzen.
completed_feature_df = live_feature_df.copy()
completed_feature_df["wind_next_hour"] = label_value

# Zeile mit gleichem Standort und Zeitpunkt aktualisieren.
wind_fg.insert(completed_feature_df, wait=True)

display(completed_feature_df)

,time,wind_current,wind_mean_6h,wind_next_hour,location
0,2026-10-04 17:00:00+00:00,8.1,7.933333,6.4,basel


In [37]:
# Daten erneut aus Hopsworks lesen.
check_df = wind_fg.read()
check_df["time"] = pd.to_datetime(check_df["time"], utc=True)

# Genau unsere ergänzte Zeile auswählen.
saved_row = check_df.loc[
    (check_df["location"] == "basel")
    & (check_df["time"] == feature_time)
]

display(saved_row)

if len(saved_row) != 1:
    raise ValueError("Es wurde nicht genau eine passende Zeile gefunden.")

if saved_row["wind_next_hour"].iloc[0] != label_value:
    raise ValueError("Der gespeicherte Zielwert stimmt nicht überein.")

print("Bestätigt: Der Zielwert ist in Hopsworks gespeichert.")

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (0.88s) 


,time,wind_current,wind_mean_6h,wind_next_hour,location
0,2026-10-04 17:00:00+00:00,8.1,7.933333,6.4,basel


Bestätigt: Der Zielwert ist in Hopsworks gespeichert.
